In [74]:
import pandas as pd

data = pd.read_csv("../../data/raw/kathmandu_full_raw_2023_2024.csv")

Datetime format conversion and sorting (although already sorted)

In [75]:

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)
data.head(3)

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2023-01-01 00:00:00,83.4,119.2,1689,38.0,12.9,67,7.9,88,1.1,162,872.6
1,2023-01-01 01:00:00,82.2,117.6,1594,31.5,13.2,70,8.0,85,2.9,150,872.3
2,2023-01-01 02:00:00,80.8,115.7,1504,24.6,13.6,73,8.0,81,4.1,135,871.9


Create target column. For instance i, target is PM2.5 of instance (i+1) i.e. PM2.5 after 1 hour.

In [76]:
data["target"] = data["pm2_5"].shift(-1)
data.head(3)

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,target
0,2023-01-01 00:00:00,83.4,119.2,1689,38.0,12.9,67,7.9,88,1.1,162,872.6,82.2
1,2023-01-01 01:00:00,82.2,117.6,1594,31.5,13.2,70,8.0,85,2.9,150,872.3,80.8
2,2023-01-01 02:00:00,80.8,115.7,1504,24.6,13.6,73,8.0,81,4.1,135,871.9,78.2


PM2.5 lag features. i.e. PM2.5 at (t-1), (t-2), ... (t-12)

In [77]:
for i in range(1, 13):
    data[f"pm2_5_lag_{i}"] = data["pm2_5"].shift(i)

data[20:25]

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,...,pm2_5_lag_3,pm2_5_lag_4,pm2_5_lag_5,pm2_5_lag_6,pm2_5_lag_7,pm2_5_lag_8,pm2_5_lag_9,pm2_5_lag_10,pm2_5_lag_11,pm2_5_lag_12
20,2023-01-01 20:00:00,88.2,127.1,1713,52.2,11.1,62,9.2,81,3.1,...,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3,71.5,88.5
21,2023-01-01 21:00:00,95.2,136.9,1845,54.2,11.4,56,8.3,85,3.6,...,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3,71.5
22,2023-01-01 22:00:00,96.6,138.8,1921,53.5,11.8,52,7.4,89,2.0,...,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3
23,2023-01-01 23:00:00,93.1,134.1,1918,48.8,12.1,53,6.7,92,2.0,...,88.2,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8
24,2023-01-02 00:00:00,90.3,129.9,1795,41.6,12.6,59,6.0,93,2.5,...,95.2,88.2,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2


Pollutant lag features, for pollutants, (t-1), (t-2), (t-3).

In [78]:
pollutants = ["pm10", "carbon_monoxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone"]

for col in pollutants:
    for i in range(1, 4):
        data[f"{col}_lag_{i}"] = data[col].shift(i)

Meterological lag features, (t-1).

In [79]:
meteo = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "surface_pressure"]

for col in meteo:
    data[f"{col}_lag_1"] = data[col].shift(1)

Meterological bin features. (based on meterological analysis in EDA)

In [80]:
import numpy as np

data["temp_bin"] = pd.cut(
    data["temperature_2m"],
    bins=[-np.inf, 10, 20, np.inf],
    labels=["low", "medium", "high"]
)

data["humidity_bin"] = pd.cut(
    data["relative_humidity_2m"],
    bins=[-np.inf, 40, np.inf],
    labels=["low", "normal"]
)

data["wind_speed_bin"] = pd.cut(
    data["wind_speed_10m"],
    bins=[-np.inf, 3, 6, 12, 15, np.inf],
    labels=["calm", "light", "moderate", "strong", "very_strong"]
)

data["wind_direction_bin"] = np.where(
    (data["wind_direction_10m"] > 120) & (data["wind_direction_10m"] <= 240),
    "mid",
    "outer"
)

data["surface_pressure_bin"] = pd.cut(
    data["surface_pressure"],
    bins=[-np.inf, 865, 870, 875, np.inf],
    labels=["very_low", "low", "normal", "high"]
)

Temporal Features

In [81]:
data["hour"] = data["time"].dt.hour
data["day_of_week"] = data["time"].dt.dayofweek
data["month"] = data["time"].dt.month

Temporal bin features. (based on temporal analysis in EDA)

In [84]:
data["season"] = np.where(
    data["month"].isin([11, 12, 1, 2]),
    "Nov_Feb",
    np.where(
        data["month"].isin([3, 4, 5, 6]),
        "Mar_Jun",
        "Jul_Oct"
    )
)

data["time_of_day"] = np.where(
    data["hour"].isin([19, 20, 21, 22, 23, 0]),
    "7pm_to_12am",
    np.where(
        data["hour"].isin([1, 2, 3, 4, 5, 6, 7, 8]),
        "1am_to_8am",
        np.where(
            data["hour"].isin([9, 10, 11, 12, 13, 14, 15, 16]),
            "9am_to_4pm",
            "5pm_to_6pm"
        )
    )
)

In [85]:
data[10:15]

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,...,temp_bin,humidity_bin,wind_speed_bin,wind_direction_bin,surface_pressure_bin,hour,day_of_week,month,season,time_of_day
10,2023-01-01 10:00:00,70.3,103.6,1238,22.5,11.9,111,13.0,57,3.9,...,medium,normal,light,outer,high,10,6,1,Nov_Feb,9am_to_4pm
11,2023-01-01 11:00:00,69.8,103.6,596,3.8,11.4,145,13.9,52,6.5,...,medium,normal,moderate,outer,high,11,6,1,Nov_Feb,9am_to_4pm
12,2023-01-01 12:00:00,69.2,102.9,440,0.0,10.6,152,14.5,47,6.6,...,medium,normal,moderate,outer,high,12,6,1,Nov_Feb,9am_to_4pm
13,2023-01-01 13:00:00,64.0,95.7,429,0.0,9.8,150,14.8,45,8.0,...,medium,normal,moderate,outer,normal,13,6,1,Nov_Feb,9am_to_4pm
14,2023-01-01 14:00:00,62.2,92.7,504,2.8,9.0,141,14.2,49,10.7,...,medium,normal,moderate,outer,normal,14,6,1,Nov_Feb,9am_to_4pm
